# RQ4: Scalability

This notebook evaluates the scalability of the BnB solver across different instance sizes (20–60 polygons) and optimality tolerances. We analyze how many instances can be solved to optimality within the time limit and visualize example solutions.

## Setup and Data Loading

Load benchmark results for multiple optimality tolerances (eps) and root node strategies. Each run records solution bounds, optimality status, runtime, and exploration statistics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from algbench import read_as_pandas
from matplotlib.lines import Line2D
from tspn_bnb2.misc.paper_style import FULLWIDEFIGURE, init_params
from tspn_bnb2.schemas import AnnotatedInstance, AnnotatedSolution

init_params()
map_root_name = {
    "LongestEdgePlusFurthestSite": "LEFP",
    "RandomRoot": "Random",
    "OrderRoot": "CHR",
    "LongestPair": "LE",
}

In [ ]:
def read_row(row):
    instance = AnnotatedInstance.model_validate_json(
        row["parameters"]["args"]["kwargs"]["instance_json"]
    )
    solution = (
        AnnotatedSolution.model_validate_json(row["result"]["solution"])
        if row["result"]["solution"]
        else None
    )

    if solution is None:
        print(
            row["result"]["error"],
            row["parameters"]["args"]["kwargs"]["instance_name"],
            row["parameters"]["args"]["alg_params"],
        )
        return None

    # if row["parameters"]["args"]["alg_params"]["root"] == "LongestPair":
    #    return None

    instance_name = row["parameters"]["args"]["kwargs"]["instance_name"]

    return {
        "solution": solution,
        "upper_bound": solution.upper_bound,
        "lower_bound": solution.lower_bound,
        "is_optimal": solution.is_optimal,
        "relative_gap": solution.relative_gap,
        "num_discarded_sequences": solution.statistics.get("num_discarded_sequences", 0),
        "num_explored": solution.statistics.get("num_explored", 0),
        "instance_name": instance_name,
        "instance": instance,
        "instance_type": instance.derive_instance_type(),
        "solve_time": row["result"].get("solve_time"),
        "n": instance.num_polygons(),
        "eps": row["parameters"]["args"]["alg_params"]["eps"],
        "root": map_root_name[row["parameters"]["args"]["alg_params"]["root"]],
    }


benchmark = read_as_pandas("results_optimality_gaps_300", read_row)
print("loaded", len(benchmark), "runs")

In [ ]:
benchmark

## Optimality Counts by Instance Size and Tolerance

In [ ]:
opt_counts = benchmark.groupby(["n", "eps"])["is_optimal"].value_counts().unstack(fill_value=0)
print(opt_counts)

## Solution Validation

Cross-check feasibility and bound consistency across all runs sharing the same instance and tolerance.

In [ ]:
# validate that solutions are correct.
for eps in benchmark["eps"].unique():
    n_checks = 0
    current_bench = benchmark[benchmark["eps"] == eps]
    for _, row in current_bench.iterrows():
        solution = row["solution"]
        if solution is None:
            continue
        assert solution.check_feasibility(row["instance"], eps=eps)
        same_instance = current_bench[current_bench["instance_name"] == row["instance_name"]]
        for _, other in same_instance.iterrows():
            if other["solution"] is None:
                continue
            check = solution.plausibility_check(
                other["solution"],
                eps=eps + 0.01,
            )
            assert check["valid"], (
                f"Check failed for {row['instance_name']}, {check} {other['solution']}"
            )
            n_checks += 1

    print(n_checks, "checks succeeded")

## Cactus Plot: Instances Solved Over Time

Shows the cumulative number of instances solved to optimality as a function of solve time, for each combination of optimality tolerance and root strategy. The vertical dashed line marks the earliest time at which all instances were solved by any configuration.

In [ ]:
fig, ax = plt.subplots(figsize=FULLWIDEFIGURE)

earliest_all = float("inf")
df = {"solve_time": [], "num_solved": [], "eps": [], "root": []}

print(benchmark["instance_name"].nunique())

last_solved_for_algo = {
    (eps, root): 0 for eps in benchmark["eps"].unique() for root in benchmark["root"].unique()
}

for _, row in benchmark.sort_values(by=["solve_time"], ascending=True).iterrows():
    if not row["is_optimal"]:
        if row["eps"] == 0.001 and row["root"] == "OrderRoot":
            print(row["instance_name"], round((row["solution"].alg_relative_gap - 1) * 100, 2), "%")
        continue
    key = (row["eps"], row["root"])
    last_solved_for_algo[key] += 1
    df["solve_time"].append(row["solve_time"])
    df["num_solved"].append(last_solved_for_algo[key])
    df["eps"].append(row["eps"])
    df["root"].append(row["root"])

    if (
        last_solved_for_algo[key] == benchmark["instance_name"].nunique()
        and row["solve_time"] < earliest_all
    ):
        earliest_all = row["solve_time"]

print(earliest_all)

df = pd.DataFrame(df)

sns.lineplot(
    df,
    ax=ax,
    x="solve_time",
    y="num_solved",
    hue="eps",
    drawstyle="steps-pre",
    style="root",
    palette="tab10",
)

ax.set_ylabel(r"\# solved instances")
ax.set_xlabel("runtime [s]")
ax.set_ylim(300, 600)
ax.axvline(earliest_all, color="black", linestyle="dashed")

handles, labels = ax.get_legend_handles_labels()
labels[0] = "optimality tolerance"

for _ in range(3):
    handles.append(Line2D([], []))  # dummy handle
    labels.append("")  # empty label
ax.legend(handles=handles, labels=labels, ncols=2)

fig.savefig("../plots/rq4_scalability/optimality.pdf", bbox_inches="tight")

## Optimality Rate Summary

Percentage of instances solved to optimality for each (root strategy, tolerance) combination.

In [ ]:
for (eps, root), count in last_solved_for_algo.items():
    print(
        root,
        eps,
        round(count / benchmark["instance_name"].nunique() * 100, 2),
        "were solved to optimality",
    )

print("Summary of optimality rates:")
print(benchmark.groupby(["root", "eps"])["is_optimal"].mean())

print("Summary of optimality rates n<=50:")
print(benchmark[benchmark["n"] <= 50].groupby(["root", "eps"])["is_optimal"].mean())

print("Summary of optimality rates n==60:")
print(benchmark[benchmark["n"] == 60].groupby(["root", "eps"])["is_optimal"].mean())

## Loading Original Instances

Load the original (unsimplified) instances from the zip archive for visualization purposes.

In [ ]:
import zipfile
from pathlib import Path

original_instances = {}
with zipfile.ZipFile("../../instances/instances_socg.zip", "r") as zip_ref:
    for filename in zip_ref.namelist():
        if not filename.endswith(".json"):
            continue
        original_instances[Path(filename).stem] = AnnotatedInstance.model_validate_json(
            zip_ref.open(filename).read()
        )

## Example Solutions: 3x3 Grids

Visualize optimal solutions on a 3x3 grid of instances, sampling across instance types and sizes. Shows original (unsimplified) polygons with the optimal tour overlaid.

In [ ]:
from numpy.random import RandomState
from shapely.plotting import plot_line, plot_polygon

for seed, min_n in zip([1337, 1337], [20, 40]):
    fig, axs = plt.subplots(ncols=3, nrows=3, figsize=(8, 8))
    random_state = RandomState(seed)

    bench = benchmark[
        (benchmark["root"] == "CHR") & (benchmark["eps"] == 0.001) & (benchmark["is_optimal"])
    ]

    cmap = plt.get_cmap("tab20")
    i = 0

    for n in [min_n, min_n + 10, min_n + 20]:
        for instance_type in benchmark["instance_type"].unique():
            for _, entry in (
                bench[
                    (bench["n"] <= n)
                    & (bench["n"] >= max(min_n, n - 9))
                    & (bench["instance_type"] == instance_type)
                ]
                .sample(1, random_state=random_state)
                .iterrows()
            ):
                ax = axs[i // 3, i % 3]
                i += 1
                orig_instance = original_instances[entry["instance_name"]]
                for j, poly in enumerate(orig_instance.polygons):
                    color = cmap(j % cmap.N)
                    face_rgba = (*color[:3], 0.5)
                    plot_polygon(
                        poly, ax=ax, facecolor=face_rgba, edgecolor="black", add_points=False
                    )

                plot_line(
                    entry["solution"].trajectory,
                    ax=ax,
                    color="black",
                    linewidth=1.5,
                    add_points=False,
                )
                coords = np.array(entry["solution"].trajectory.coords)
                ax.scatter(coords[:, 0], coords[:, 1], color="black", s=10, zorder=5)

                n_digits = 2
                if round(entry["solve_time"], n_digits) == 0:
                    n_digits += 1

                ax.set_title(
                    f"{entry['instance_type']}\n"
                    f"$OPT={round(entry['upper_bound'], 2)}$\n"
                    f"$n={orig_instance.num_polygons()}$ ({round(entry['solve_time'], n_digits)}s)"
                )

                ax.set_aspect("equal")
                ax.set_axis_off()

    fig.tight_layout()
    fig.savefig(f"../plots/examples/grid_3x3_{min_n}_{seed}.pdf", bbox_inches="tight")

## Simplification Examples

Find instances where the optimal tour visits fewer points than there are polygons — evidence that the solver's simplification (skipping polygons whose convex hull is already spanned) is effective.

In [ ]:
# find a solution with few points but any polygons:

bench = benchmark[
    (benchmark["root"] == "OrderRoot")
    & (benchmark["n"] == 40)
    & (benchmark["eps"] == 0.001)
    & (benchmark["is_optimal"])
]
cmap = plt.get_cmap("tab20")

for _, entry in bench.iterrows():
    if len(entry["solution"].trajectory.coords) < 20:
        fig, ax = plt.subplots(figsize=(6, 6))

        orig_instance = original_instances[entry["instance_name"]]
        for j, poly in enumerate(orig_instance.polygons):
            color = cmap(j % cmap.N)
            face_rgba = (*color[:3], 0.5)
            plot_polygon(poly, ax=ax, facecolor=face_rgba, edgecolor="black", add_points=False)

        plot_line(entry["solution"].trajectory, ax=ax, color="black")

        ax.set_aspect("equal")
        ax.set_axis_off()
        fig.tight_layout()

        fig.savefig(
            f"../plots/examples/simplification/{entry['instance_name']}.pdf", bbox_inches="tight"
        )

## Larger Instance Visualizations

Export individual solution plots for all optimally solved random instances with 50+ polygons.

In [ ]:
bench = benchmark[
    (benchmark["root"] == "CHR") & (benchmark["eps"] == 0.001) & (benchmark["is_optimal"])
]

cmap = plt.get_cmap("tab20")
i = 0
print(len(bench))

for _n in [60]:
    for _, entry in bench[(bench["n"] >= 50) & (bench["instance_type"] == "random")].iterrows():
        fig, ax = plt.subplots(figsize=(6, 6))

        i += 1
        orig_instance = original_instances[entry["instance_name"]]
        for j, poly in enumerate(orig_instance.polygons):
            color = cmap(j % cmap.N)
            face_rgba = (*color[:3], 0.5)
            plot_polygon(poly, ax=ax, facecolor=face_rgba, edgecolor="black", add_points=False)

        plot_line(entry["solution"].trajectory, ax=ax, color="black")

        ax.set_aspect("equal")
        ax.set_axis_off()
        fig.tight_layout()
        fig.savefig(
            f"../plots/examples/opt_50-60/{entry['instance_name']}_{orig_instance.num_polygons()}_{round(entry['solve_time'])}s.pdf",
            bbox_inches="tight",
        )